# Programming for Artificial Intelligence
## Step-by-Step Tutorial: Prolog & OpenCV

**Level:** Introductory  

> **Big picture:** Both Prolog and OpenCV are tools for building intelligent systems — they just approach it differently. Prolog reasons *logically* ("what is true given these facts?"), while OpenCV reasons *visually* ("what does this image contain?"). As an AI programmer, you'll use both kinds of thinking.

---

## Table of Contents
1. [Part 1 — Prolog](#part1)
   - 1.1 Facts, Rules, and Queries
   - 1.2 Recursion and Backtracking
2. [Part 2 — OpenCV](#part2)
   - 2.1 Basics: Read, Display, Write, Resize, Crop
   - 2.2 Thresholding and Edge Detection
   - 2.3 Face Detection
3. [Part 3 — Cheat Sheets](#part3)


# Part 1 — Prolog <a id='part1'></a>


## 1.1 Facts, Rules, and Queries

### What is a Fact?
A **fact** is the simplest building block in Prolog. It declares something unconditionally true. Think of it as a row in a database.

```prolog
% Syntax: predicate(argument1, argument2).

parent(john, alice).    % John is a parent of Alice
parent(alice, bob).     % Alice is a parent of Bob
parent(alice, carol).   % Alice is a parent of Carol
parent(bob, dan).       % Bob is a parent of Dan
```

### What is a Rule?
A **rule** defines a relationship in terms of existing facts. Read `:-` as "if".

```prolog
% grandparent(X, Y) is true IF there exists Z
% such that X is a parent of Z AND Z is a parent of Y.

grandparent(X, Y) :-
    parent(X, Z),    % X is a parent of Z
    parent(Z, Y).    % Z is a parent of Y
```

### What is a Query?
A **query** asks Prolog to prove something or find values satisfying a condition.

```prolog
?- grandparent(john, bob).
% Output: true

?- parent(alice, X).
% Output: X = bob ;
%         X = carol

?- grandparent(GP, GC).
% Output: GP = john, GC = bob ;
%         GP = john, GC = carol ;
%         GP = alice, GC = dan
```

### Full Working Example — Save as `family.pl`

```prolog
%% === family.pl ===
%% Load with: ?- consult('family.pl').  OR  ?- [family].

%% --- FACTS ---
parent(john, alice).
parent(alice, bob).
parent(alice, carol).
parent(bob, dan).

%% --- RULES ---
grandparent(X, Y) :-
    parent(X, Z),
    parent(Z, Y).

% A sibling shares at least one parent, and is not the same person
sibling(X, Y) :-
    parent(P, X),
    parent(P, Y),
    X \= Y.        % \= means "not equal"

%% --- SAMPLE QUERIES ---
% ?- grandparent(john, bob).    → true
% ?- sibling(bob, carol).       → true
% ?- sibling(bob, bob).         → false
% ?- parent(alice, W).          → W = bob ; W = carol
```


## 1.2 Recursion and Backtracking

### Recursion — Factorial
Prolog recursion uses a **base case** (stops the recursion) and a **recursive case** (reduces the problem).

```prolog
%% === factorial.pl ===

% BASE CASE: factorial of 0 is 1
factorial(0, 1).

% RECURSIVE CASE
factorial(N, F) :-
    N > 0,               % Guard: only proceed if N is positive
    N1 is N - 1,         % Compute N-1, bind to N1
    factorial(N1, F1),   % Recursively compute factorial(N-1)
    F is N * F1.         % F = N * result
```

```prolog
?- factorial(5, X).
% Output: X = 120

?- factorial(0, X).
% Output: X = 1
```

**Trace of `factorial(3, X)`:**
```
factorial(3, F)
  → 3 > 0 ✓, N1 = 2
  → factorial(2, F1)
      → 2 > 0 ✓, N1 = 1
      → factorial(1, F1)
          → 1 > 0 ✓, N1 = 0
          → factorial(0, F1)  ← BASE CASE: F1 = 1
          → F = 1 * 1 = 1
      → F = 2 * 1 = 2
  → F = 3 * 2 = 6
```

### Backtracking — List Membership
**Backtracking** is Prolog's built-in search. When there are multiple ways to prove something, Prolog tries them one at a time. If one path fails, it backtracks and tries the next.

```prolog
%% === member.pl ===

% BASE CASE: X is a member if it is the HEAD of the list
member(X, [X|_]).

% RECURSIVE CASE: X is a member if it is in the TAIL
member(X, [_|T]) :-
    member(X, T).
```

```prolog
?- member(3, [1, 2, 3, 4]).
% Output: true

?- member(5, [1, 2, 3, 4]).
% Output: false

?- member(X, [a, b, c]).
% Output: X = a ;
%         X = b ;
%         X = c
```

**How backtracking works:**
```
Query: member(3, [1, 2, 3, 4])

Try rule 1: member(3, [1|_])  → 3 = 1? NO. Backtrack.
Try rule 2: member(3, [2, 3, 4])
  Try rule 1: member(3, [2|_]) → 3 = 2? NO. Backtrack.
  Try rule 2: member(3, [3, 4])
    Try rule 1: member(3, [3|_]) → 3 = 3? YES ✓
Result: true
```

> **AI connection:** Backtracking is essentially *search with pruning* — the same idea used in game trees, constraint satisfaction, and planning algorithms.

---
# Part 2 — OpenCV <a id='part2'></a>

Reference: [GeeksforGeeks OpenCV Python Tutorial](https://www.geeksforgeeks.org/opencv-python-tutorial/)

## Setup — Install OpenCV
Run this cell once to install OpenCV.

In [ ]:
# Install OpenCV (run once)
!pip install opencv-python

## 2.1 Basics: Read, Display, Write, Resize, Crop

### Generate a Sample Test Image
Since you may not have an image file ready, this cell creates one for you to use throughout the notebook.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Create a simple synthetic test image (400x300, 3 channels)
sample = np.zeros((300, 400, 3), dtype=np.uint8)

# Add colored rectangles
sample[0:150, 0:200]   = [255, 100, 50]    # Top-left: blue-ish
sample[0:150, 200:400] = [50, 200, 100]    # Top-right: green-ish
sample[150:300, 0:200] = [100, 50, 200]    # Bottom-left: red-ish
sample[150:300, 200:400] = [200, 200, 50]  # Bottom-right: cyan-ish

# Draw a circle and text
cv2.circle(sample, (200, 150), 60, (255, 255, 255), -1)  # White filled circle
cv2.putText(sample, 'AI Lab', (145, 160),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

# Save it as photo.jpg so we can use it throughout
cv2.imwrite('photo.jpg', sample)
print("Sample image 'photo.jpg' created successfully!")

# Display it
plt.figure(figsize=(5, 4))
plt.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
plt.title('Sample Test Image')
plt.axis('off')
plt.show()

### Reading, Displaying, and Writing Images

> **Prolog parallel:** `cv2.imread()` is like *loading your knowledge base* — you bring external data into your program's working memory before you can reason about it.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# --- READ an image from disk ---
# cv2.imread(path, flag)
# flag=1  → color (BGR)    flag=0 → grayscale    flag=-1 → with alpha channel
img = cv2.imread('photo.jpg', 1)

# IMPORTANT: OpenCV loads images in BGR (Blue-Green-Red) order.
# Matplotlib expects RGB. Convert before displaying!
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# --- DISPLAY using matplotlib (correct for Jupyter notebooks) ---
plt.figure(figsize=(5, 4))
plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')
plt.show()

print(f"Image shape: {img.shape}")   # (height, width, channels)
print(f"Data type  : {img.dtype}")   # uint8 (values 0-255)

# --- WRITE / SAVE to disk ---
cv2.imwrite('output.jpg', img)       # Saves in BGR format (correct for files)
print("Image saved as 'output.jpg'")

# --- NOTE on cv2.imshow ---
# cv2.imshow() opens a pop-up window — use it in standalone .py scripts only.
# In Jupyter, always use matplotlib instead.
#
# Standalone script usage (for reference):
# cv2.imshow('Window Title', img)
# cv2.waitKey(0)             # Wait for any key press
# cv2.destroyAllWindows()    # Close all OpenCV windows

### Resizing Images

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('photo.jpg')

# --- RESIZE: specify exact dimensions ---
# cv2.resize(src, (width, height))
# NOTE: OpenCV uses (width, height) but img.shape returns (height, width, channels)
resized = cv2.resize(img, (300, 200))

# --- RESIZE: scale by a factor ---
# fx=0.5 → 50% of original width, fy=0.5 → 50% of original height
half = cv2.resize(img, None, fx=0.5, fy=0.5)

# --- RESIZE: double the size ---
double = cv2.resize(img, None, fx=2.0, fy=2.0)

print(f"Original : {img.shape}")
print(f"Resized  : {resized.shape}")
print(f"Half     : {half.shape}")
print(f"Double   : {double.shape}")

# Display
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, image, title in zip(axes,
    [img, resized, half],
    [f'Original {img.shape[1]}x{img.shape[0]}',
     f'Resized 300x200',
     f'Half size {half.shape[1]}x{half.shape[0]}']):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Cropping with Array Slicing

OpenCV images are NumPy arrays. Cropping is simply NumPy array slicing.

**Syntax:** `img[y_start:y_end, x_start:x_end]`  
Remember: rows = Y (height direction), columns = X (width direction)

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('photo.jpg')
h, w = img.shape[:2]   # Get height and width
print(f"Image dimensions: height={h}, width={w}")

# --- CROP: top-left quarter ---
top_left = img[0:h//2, 0:w//2]

# --- CROP: center region ---
y1, y2 = h//4, 3*h//4
x1, x2 = w//4, 3*w//4
center = img[y1:y2, x1:x2]

# --- CROP: bottom-right quarter ---
bottom_right = img[h//2:h, w//2:w]

# Display all crops
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, image, title in zip(axes,
    [img, top_left, center, bottom_right],
    ['Original', 'Top-left quarter', 'Center crop', 'Bottom-right quarter']):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 2.2 Thresholding and Edge Detection

### Image Thresholding

Thresholding converts a grayscale image to **binary** (black and white) by comparing each pixel to a threshold value.

- Pixels **above** the threshold → white (255)
- Pixels **below** the threshold → black (0)

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('photo.jpg')

# Must convert to grayscale before thresholding
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- SIMPLE THRESHOLDING ---
# cv2.threshold(src, thresh, maxval, type)
# Returns: (threshold_value_used, result_image)

# THRESH_BINARY: pixels > 127 → 255 (white), pixels ≤ 127 → 0 (black)
ret, thresh_binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# THRESH_BINARY_INV: inverted version
ret, thresh_inv = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)

# THRESH_TRUNC: pixels > 127 are capped at 127, others unchanged
ret, thresh_trunc = cv2.threshold(gray, 127, 255, cv2.THRESH_TRUNC)

print(f"Threshold used: {ret}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, image, title in zip(axes,
    [gray, thresh_binary, thresh_inv, thresh_trunc],
    ['Grayscale', 'THRESH_BINARY\n(t=127)', 'THRESH_BINARY_INV\n(t=127)', 'THRESH_TRUNC\n(t=127)']):
    ax.imshow(image, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Adaptive Thresholding

A single global threshold fails when lighting is uneven (e.g., a photo under a lamp). **Adaptive thresholding** calculates the threshold *locally* for each region.

> **Prolog parallel:** Just like Prolog evaluates rules in the context of local variable bindings (not globally), adaptive thresholding makes decisions based on the *local neighbourhood* of each pixel.

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('photo.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Global threshold for comparison
_, global_thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# --- ADAPTIVE THRESHOLDING ---
# cv2.adaptiveThreshold(src, maxval, adaptiveMethod, thresholdType, blockSize, C)
#
# adaptiveMethod:
#   ADAPTIVE_THRESH_MEAN_C     → threshold = mean of neighbourhood
#   ADAPTIVE_THRESH_GAUSSIAN_C → threshold = Gaussian-weighted mean (better for photos)
#
# blockSize: size of the neighbourhood (must be ODD, e.g. 11)
# C: constant subtracted from the computed mean

adaptive_mean = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    11, 2
)

adaptive_gaussian = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11, 2
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, image, title in zip(axes,
    [gray, global_thresh, adaptive_mean, adaptive_gaussian],
    ['Grayscale', 'Global threshold\n(t=127)',
     'Adaptive Mean\n(block=11, C=2)', 'Adaptive Gaussian\n(block=11, C=2)']):
    ax.imshow(image, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Edge Detection with Canny

The Canny algorithm finds edges (sharp brightness boundaries) in an image. It is one of the most important tools in computer vision.

**The Canny algorithm works in 4 steps:**
1. **Noise reduction** — Gaussian blur smooths the image  
2. **Gradient calculation** — finds where intensity changes sharply  
3. **Non-maximum suppression** — thins edges to 1-pixel width  
4. **Hysteresis thresholding** — uses two thresholds to keep only real edges  



In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('photo.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- STEP 1: Blur to reduce noise (recommended before Canny) ---
# GaussianBlur(src, ksize, sigmaX)
# ksize = kernel size — must be odd. (5,5) is a common starting point.
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# --- STEP 2: Apply Canny edge detection ---
# cv2.Canny(image, threshold1, threshold2)
#
# threshold1 (lower): edges below this are DISCARDED
# threshold2 (upper): edges above this are KEPT
# Edges between the two are kept ONLY if connected to a strong edge.
#
# Low thresholds → sensitive, detects many edges (including noise)
# High thresholds → strict, detects only strong clear edges

edges_low    = cv2.Canny(blurred, 30, 100)   # Sensitive
edges_medium = cv2.Canny(blurred, 70, 150)   # Balanced
edges_high   = cv2.Canny(blurred, 100, 200)  # Strict

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, image, title in zip(axes,
    [gray, edges_low, edges_medium, edges_high],
    ['Grayscale', 'Canny\n(low: 30, 100)',
     'Canny\n(medium: 70, 150)', 'Canny\n(high: 100, 200)']):
    ax.imshow(image, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Count edge pixels in each result
for name, edges in [('Low', edges_low), ('Medium', edges_medium), ('High', edges_high)]:
    count = cv2.countNonZero(edges)
    print(f"{name} threshold: {count} edge pixels detected")

---
## 2.3 Face Detection with CascadeClassifier

OpenCV includes pre-trained **Haar Cascade** classifiers — XML files that encode what faces look like. The classifier scans an image at multiple scales to find faces of any size.

**How the Haar Cascade works:**
- Trained on thousands of face and non-face images
- Looks for patterns: dark eye regions, bright nose bridge, bright cheeks, etc.
- Scans at multiple scales (faces can be near or far from the camera)
- `minNeighbors` filters false positives — a detection must appear in multiple nearby positions



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- CREATE A SYNTHETIC TEST IMAGE WITH FACE-LIKE SHAPES ---
# (Replace 'face_photo.jpg' with a real photo for best results)
# This cell demonstrates the full pipeline even without a real photo.

# Check if a real photo exists; otherwise create a placeholder
import os
if os.path.exists('face_photo.jpg'):
    img = cv2.imread('face_photo.jpg')
    print("Loaded 'face_photo.jpg'")
else:
    # Create a synthetic placeholder
    img = np.ones((400, 600, 3), dtype=np.uint8) * 200
    # Draw a simple face outline
    cv2.ellipse(img, (300, 200), (80, 100), 0, 0, 360, (180, 140, 110), -1)  # Head
    cv2.ellipse(img, (270, 185), (15, 10), 0, 0, 360, (80, 60, 40), -1)      # Left eye
    cv2.ellipse(img, (330, 185), (15, 10), 0, 0, 360, (80, 60, 40), -1)      # Right eye
    cv2.ellipse(img, (300, 240), (30, 15), 0, 0, 180, (120, 60, 60), 3)      # Mouth
    print("Using synthetic placeholder image (no face_photo.jpg found)")
    print("TIP: Replace with a real photo named 'face_photo.jpg' for actual detection")

# Display the source image
plt.figure(figsize=(6, 5))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('Source Image for Face Detection')
plt.axis('off')
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

# --- STEP 1: Load the pre-trained Haar Cascade classifier ---
# cv2.data.haarcascades gives the path to OpenCV's built-in XML files
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

if face_cascade.empty():
    print("ERROR: Could not load cascade classifier. Check your OpenCV installation.")
else:
    print(f"Cascade loaded from: {cascade_path}")

# --- STEP 2: Convert to grayscale (cascade works on grayscale images) ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- STEP 3: Detect faces ---
# detectMultiScale(image, scaleFactor, minNeighbors, minSize)
#
# scaleFactor  : how much to shrink image each pass (1.1 = 10% smaller each time)
#                smaller value = more thorough but slower
# minNeighbors : how many detections needed to confirm a face
#                higher = fewer false positives, but may miss some faces
# minSize      : minimum face size in pixels (ignore very small detections)

faces = face_cascade.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30, 30)
)

print(f"Number of faces detected: {len(faces)}")
if len(faces) > 0:
    for i, (x, y, w, h) in enumerate(faces):
        print(f"  Face {i+1}: x={x}, y={y}, width={w}, height={h}")

# --- STEP 4: Draw bounding boxes on a copy of the image ---
img_result = img.copy()   # Always work on a copy, not the original

for (x, y, w, h) in faces:
    # Draw green rectangle around each face
    # cv2.rectangle(img, top-left, bottom-right, color_BGR, thickness)
    cv2.rectangle(img_result, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Add label text above the box
    # cv2.putText(img, text, origin, font, scale, color, thickness)
    cv2.putText(img_result, 'Face',
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 255, 0), 2)

# --- STEP 5: Display results ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_result, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Face Detection Result ({len(faces)} face(s))')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# --- BONUS: Try different minNeighbors values and compare ---
import cv2
import matplotlib.pyplot as plt

cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, neighbors in zip(axes, [1, 3, 7]):
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1,
        minNeighbors=neighbors, minSize=(20, 20)
    )
    temp = img.copy()
    for (x, y, w, h) in faces:
        cv2.rectangle(temp, (x, y), (x+w, y+h), (0, 255, 0), 2)
    ax.imshow(cv2.cvtColor(temp, cv2.COLOR_BGR2RGB))
    ax.set_title(f'minNeighbors={neighbors}\n{len(faces)} detection(s)')
    ax.axis('off')

plt.suptitle('Effect of minNeighbors on face detection sensitivity', y=1.02)
plt.tight_layout()
plt.show()
print("Lower minNeighbors = more sensitive (more false positives)")
print("Higher minNeighbors = more strict (fewer false positives, may miss faces)")

---
# Part 3 — Cheat Sheets <a id='part3'></a>

## Prolog Cheat Sheet

| Concept | Syntax | Example |
|---|---|---|
| Fact | `pred(arg).` | `parent(john, alice).` |
| Rule | `head :- body.` | `grandparent(X,Y) :- parent(X,Z), parent(Z,Y).` |
| Query | `?- goal.` | `?- grandparent(john, bob).` |
| Variable | Starts uppercase | `X`, `Y`, `Parent` |
| Anonymous var | `_` | `member(X, [X|_]).` |
| Arithmetic | `is` | `F is N * F1` |
| Not equal | `\=` | `X \= Y` |
| List head/tail | `[H|T]` | `member(X, [X|_]).` |

## OpenCV Cheat Sheet

| Task | Function | Notes |
|---|---|---|
| Read image | `cv2.imread('file.jpg')` | Returns BGR array |
| Write image | `cv2.imwrite('out.jpg', img)` | |
| Show (script) | `cv2.imshow('title', img)` + `cv2.waitKey(0)` + `cv2.destroyAllWindows()` | Not for Jupyter |
| Show (Jupyter) | `plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))` | |
| Grayscale | `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` | |
| Resize exact | `cv2.resize(img, (width, height))` | |
| Resize scale | `cv2.resize(img, None, fx=0.5, fy=0.5)` | |
| Crop | `img[y1:y2, x1:x2]` | NumPy slicing |
| Threshold | `cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)` | Returns `(retval, img)` |
| Adaptive thresh | `cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)` | |
| Blur | `cv2.GaussianBlur(gray, (5,5), 0)` | Before Canny |
| Edge detection | `cv2.Canny(gray, 100, 200)` | |
| Load cascade | `cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')` | |
| Detect faces | `cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)` | |
| Draw rectangle | `cv2.rectangle(img, (x,y), (x+w,y+h), (0,255,0), 2)` | |
| Draw text | `cv2.putText(img, 'text', (x,y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)` | |

---

**Reference:** [GeeksforGeeks OpenCV Python Tutorial](https://www.geeksforgeeks.org/opencv-python-tutorial/)